# ACTIVIDAD SESIÓN 5: INTRODUCCIÓN A MACHINE LEARNING ESCALABLE

Una tienda de cosmética quiere desarrollar un sistema inteligente que clasifique productos de *skin care* en diferentes categorías según sus características.

**Objetivo**  
Entrenar un modelo de clasificación con **MLlib** que prediga el tipo de piel recomendado para cada producto.

**Dataset**: `skincare_products.csv`

## 1. Carga y exploración de datos (2 puntos)
- Cargar los datos desde `skincare_products.csv` en un DataFrame de PySpark.  
- Mostrar las primeras filas del dataset.  
- Realizar un resumen estadístico de las variables numéricas.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


Origen de datos: archivo skincare_products.csv
+--------------------+-----------+---------+---+------------+
|        Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|
+--------------------+-----------+---------+---+------------+
|   Ácido Hialurónico|       Alto|    Medio|  0|        Seco|
|             Retinol|       Bajo|     Alto|  0|       Graso|
|          Vitamina C|      Medio|    Medio| 30|       Mixto|
|           Aloe Vera|       Alto|     Bajo| 15|    Sensible|
|         Niacinamida|      Medio|    Medio|  0|       Mixto|
|           Ceramidas|       Alto|     Bajo|  0|        Seco|
|    Ácido Salicílico|       Bajo|     Alto|  0|       Graso|
|   Centella Asiática|      Medio|    Medio| 20|    Sensible|
|Extracto de Té Verde|      Medio|     Alto|  0|       Mixto|
|   Manteca de Karité|       Alto|     Bajo|  0|        Seco|
| Extracto de Regaliz|      Medio|    Medio| 15|    Sensible|
|          Vitamina E|       Alto|    Medio| 25|        Seco|
|           Bakuchiol| 

In [18]:

spark = SparkSession.builder.appName("SkinCareML").getOrCreate()

# Intentar cargar dataset

df = spark.read.csv("skincare_products.csv", header=True, inferSchema=True)
origen = "archivo skincare_products.csv"



In [19]:

print(f"Origen de datos: {origen}")
df.show(5)
df.describe().show()

Origen de datos: archivo skincare_products.csv
+-----------------+-----------+---------+---+------------+
|     Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|
+-----------------+-----------+---------+---+------------+
|Ácido Hialurónico|       Alto|    Medio|  0|        Seco|
|          Retinol|       Bajo|     Alto|  0|       Graso|
|       Vitamina C|      Medio|    Medio| 30|       Mixto|
|        Aloe Vera|       Alto|     Bajo| 15|    Sensible|
|      Niacinamida|      Medio|    Medio|  0|       Mixto|
+-----------------+-----------+---------+---+------------+
only showing top 5 rows
+-------+----------------+-----------+---------+-----------------+------------+
|summary|    Ingredientes|Hidratación|Absorción|              SPF|Tipo de Piel|
+-------+----------------+-----------+---------+-----------------+------------+
|  count|              20|         20|       20|               20|          20|
|   mean|            NULL|       NULL|     NULL|              7.5|        NULL

## 2. Preprocesamiento de datos (2 puntos)
- Convertir la columna **"Tipo de Piel"** en valores numéricos (0, 1, 2, 3).  
- Transformar las variables categóricas (**Hidratación** y **Absorción**) en valores numéricos:  
  - Hidratación: Bajo (0), Medio (1), Alto (2)  
  - Absorción: Bajo (0), Medio (1), Alto (2)  
- Unir todas las características en un vector usando `VectorAssembler`.

In [20]:
from pyspark.ml.feature import VectorAssembler

# Mapear Tipo de Piel manualmente
df = df.withColumn("label",
    F.when(F.col("tipo_piel")=="Seco", 0)
     .when(F.col("tipo_piel")=="Graso", 1)
     .when(F.col("tipo_piel")=="Mixto", 2)
     .when(F.col("tipo_piel")=="Sensible", 3)
)

# Mapear Hidratación y Absorción
map_hid = {"Bajo":0, "Medio":1, "Alto":2}
map_abs = {"Bajo":0, "Medio":1, "Alto":2}

df = df.replace(map_hid, subset=["hidratacion"])       .replace(map_abs, subset=["absorcion"])       .withColumnRenamed("hidratacion", "hidratacion_num")       .withColumnRenamed("absorcion", "absorcion_num")


{"ts": "2025-09-08 21:58:55.959", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703", "context": {"file": "line 5 in cell [20]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o292.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703;\n'Project [Ingredientes#3878, Hidratación#3879, Absorción#3880, SPF#3881, Tipo de Piel#3882, CASE WHEN '`=`('tipo_piel, Seco) THEN 0 WHEN '`=`('tipo_p

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703;
'Project [Ingredientes#3878, Hidratación#3879, Absorción#3880, SPF#3881, Tipo de Piel#3882, CASE WHEN '`=`('tipo_piel, Seco) THEN 0 WHEN '`=`('tipo_piel, Graso) THEN 1 WHEN '`=`('tipo_piel, Mixto) THEN 2 WHEN '`=`('tipo_piel, Sensible) THEN 3 END AS label#4249]
+- Relation [Ingredientes#3878,Hidratación#3879,Absorción#3880,SPF#3881,Tipo de Piel#3882] csv


25/09/08 23:30:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 657986 ms exceeds timeout 120000 ms
25/09/08 23:30:09 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/08 23:30:10 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o

In [ ]:

# Ensamblar features
assembler = VectorAssembler(
    inputCols=["hidratacion_num", "absorcion_num", "spf"],
    outputCol="features"
)
df_ready = assembler.transform(df)
df_ready.select("ingredientes","features","label").show(truncate=False)

## 3. División de datos y entrenamiento del modelo (3 puntos)
- Dividir los datos en **80% entrenamiento** y **20% prueba**.  
- Entrenar un modelo de **Árboles de Decisión** con MLlib usando las características del dataset.

## 4. Predicción y evaluación (2 puntos)
- Aplicar el modelo al conjunto de prueba y mostrar las predicciones.  
- Calcular la **precisión del modelo** usando `MulticlassClassificationEvaluator`.

## 5. Análisis de resultados y mejoras (1 punto)
- Explicar en 3–5 líneas qué tan preciso fue el modelo y cómo se podría mejorar  
  (por ejemplo, usando otro algoritmo o ajustando parámetros).

## INSTRUCCIONES ADICIONALES
- Incluye un documento con el análisis de tus resultados y mejoras.  
- **Puntos totales = 10**.  
- Comprimir el archivo en formato `.zip` o `.rar`.  
- Subir el archivo a la plataforma.